# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, inspecting, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library and Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s from the dataset.

In [ ]:
# List all available record sets and their field IDs
print("Available record sets in the dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set @id: {record_set['@id']} | Name: {record_set.get('name', '-')}" )
    if 'fields' in record_set:
        print("    Fields:")
        for field in record_set['fields']:
            print(f"      - Field @id: {field['@id']}, Name: {field.get('name', '')}, DataType: {field.get('dataType', '')}")
    print("")
# Save the first record set id for use in the rest of the notebook
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
if len(record_set_ids) == 0:
    print("No record sets found in this Croissant metadata.")

## 3. Data Extraction
Load data from each available record set into a dataframe. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract all available record sets as dataframes, using their `@id`

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set: {rs_id}")
    if len(df.columns) > 0:
        print(f"Columns: {df.columns.tolist()[:8]} {'...' if len(df.columns) > 8 else ''}")
    print()

# For demonstration, select the first record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id is not None:
    print(f"Using Record Set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set to display.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data exploration: filter on a numeric field, normalize, group by a demographic or other field using `@id`.

In [ ]:
# Identify likely numeric and grouping fields by their @id (see previous overview code cell output).
import numpy as np

# Example field IDs (replace with ones present in your dataset record set):
# Replace with the actual @id for PHQ-9 Total Score (e.g., 'phq9_total_score') and Village (e.g., 'village')

numeric_field_id = None
group_field_id = None

if main_record_set_id is not None:
    candidate_numeric_fields = [c for c in dataframes[main_record_set_id].columns if 'phq' in c.lower() or 'score' in c.lower() or 'age' in c.lower()]
    if len(candidate_numeric_fields):
        numeric_field_id = candidate_numeric_fields[0]
    candidate_group_fields = [c for c in dataframes[main_record_set_id].columns if 'village' in c.lower() or 'gender' in c.lower() or 'education' in c.lower()]
    if len(candidate_group_fields):
        group_field_id = candidate_group_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    # Only run if both available
    df = dataframes[main_record_set_id]
    if numeric_field_id is not None and df[numeric_field_id].dtype in [np.float64, np.int64, float, int]:
        # Remove missing/NaN
        filtered_df = df[df[numeric_field_id] > 10].copy()
        print(f"Records with {numeric_field_id} > 10: {len(filtered_df)}")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name=f"{numeric_field_id}_mean")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize the distribution of a survey score and its variation by village (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if main_record_set_id is not None and numeric_field_id is not None and numeric_field_id in dataframes[main_record_set_id].columns:
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

# Boxplot by group (if possible)
if group_field_id is not None and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_record_set_id].dropna(subset=[group_field_id, numeric_field_id]))
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mental health dataset from Kilifi County, Kenya via its Croissant schema. We listed available record sets, examined their fields (by `@id`), loaded and processed the main survey record set, and performed exploratory data analysis including normalization and group statistics. Visualizations showed the distribution of a key numeric score as well as its variation by demographic. For further analysis, consult the data dictionary in the dataset documentation for exact meaning of each field `@id`.
